# Machine Learning - Exercise 2 
# Bayesian Learning

See description of the example in Russel & Norvig: Artificial Intelligence: A modern approach. Chap. 20.

In [ ]:
import numpy as np

We have 5 kinds of bags containing 2 kinds of candies (lime, cherry) in different proportions. 

10% of the bags are h1: 100% cherry, 0% lime
20% of the bags are h2: 75% cherry, 25% lime
40% of the bags are h3: 50% cherry, 50% lime
20% of the bags are h4: 25% cherry, 75% lime
10% of the bags are h5: 0% cherry, 100% lime

We choose a random bag (without knowing which type of bag is it) and extract some candies from it.
What kind of bag is it?
What is the probability of extracting a candy of a specific flavor next?

Reorganize the data:

1. Prior probability distributions P(h): probability of each hypothesis (= bag)

<center>P(h) = <0.1, 0.2, 0.4, 0.2, 0.1></center>

2. Likelihood for lime candies P(lime | H): probability of extracting a lime candy given the hypothesis space (kinda like you consider different possible scenarios and you compute the probability of an event to happen for each of them)

<center>P(lime | H) = <0, 0.25, 0.5, 0.75, 1></center> 

3. Likelihood for cherry candies P(cherry | H): 1 - likelihood for lime candies

<center>P(cherry | H) = <1, 0.55, 0.5, 0.25, 0></center> 

About the questions:

1. We choose a random bag (without knowing which type of bag is it) and extract some candies from it. What kind of bag is it?What kind of bag is it?

  We are asked to find the maxium likelihood hypothesis.
  The learner consider many learning scenarios and is interested in finding the 
  most probable hypothesis h given the observed data D.

  <center>h_map = argmax P(h | D) = argmax P(D | h)P(h)</center>
  <center>h_map = argmax probability_of_h_given_data</center>

  if we don't have info about prior probabilities we can assume each hypothesis is equally probable and thus further simplify the notation

  <center>h_map = argmax P(D | h)P(h) = argmax P(D | h)</center>

  We are interested in this shit

  In this exercise we have information about the prior probabilities.
  
  After first extraction, supposing a lime candy is extracted:

    D = {lime}

    P(h1 | D) = P(D | h1) P(h1) = 0 * 0.1 = 0

    P(h2 | D) = P(D | h2) P(h2) = 0.25 * 0.2 = 0.05
    
    P(h3 | D) = P(D | h3) P(h3) = 0.5 * 0.4 = 0.2
    
    P(h4 | D) = P(D | h4) P(h4) = 0.75 * 0.2 = 0.15
    
    P(h5 | D) = P(D | h5) P(h5) = 1 * 0.1 = 0.1

    equivalently:
    
    P(H | D) = P(D | H) P(H) = <0, 0.25, 0.5, 0.75, 1><0.1, 0.2, 0.4, 0.2, 0.1> = <0, 0.05, 0.2, 0.15, 0.1>


  2. after two extraction, suppose we extract another lime candy:

    D = {lime, lime}

    Subsequent extractions are independent from each other, we can apply the chain rule.

    P(h | D) = Product(di | h) * P(h) 

    P(hi | {lime, lime}) = P({lime, lime} | hi) P(hi) = P({lime} | hi)P({lime} | hi) P(hi) = ...

## Prior knowledge

In [3]:
PH = np.array([0.1, 0.2, 0.4, 0.2, 0.1])

PdH = {}
PdH['l'] = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
PdH['c'] = 1 - PdH['l']

print('P(H) = %s' %(str(PH)))
print('P(l|H) = %s' %(str(PdH['l'])))
print('P(c|H) = %s' %(str(PdH['c'])))

cP = PdH['l'] * PH # P(H | D) = P(D | H) P(H) Bayes rule
Pl = np.sum(cP) # P(A v B) = P(A) + P(B) - P(A ^ B) with P(A ^ B) = 0
print('P(l) = sum %s = %.3f' %(str(cP),Pl))

P(H) = [0.1 0.2 0.4 0.2 0.1]
P(l|H) = [0.   0.25 0.5  0.75 1.  ]
P(c|H) = [1.   0.75 0.5  0.25 0.  ]
P(l) = sum [0.   0.05 0.2  0.15 0.1 ] = 0.500


## Dataset

In [4]:
D = ['l','l','l','l','l']

## Bayesian Learning

In [6]:
np.set_printoptions(formatter={'float': '{: 0.3f}'.format})
P = PH
db = ''
print('P(H)      \t= %s' %(str(PH)))
for d in D: # at each extraction specified in the vector
    P = P * PdH[d]
    P = P / np.sum(P) # normalization
    db = db+d
    print('P(H|%s)  \t= %s' %(db,str(P)))

P(H)      	= [ 0.100  0.200  0.400  0.200  0.100]
P(H|l)  	= [ 0.000  0.100  0.400  0.300  0.200]
P(H|ll)  	= [ 0.000  0.038  0.308  0.346  0.308]
P(H|lll)  	= [ 0.000  0.013  0.211  0.355  0.421]
P(H|llll)  	= [ 0.000  0.004  0.132  0.335  0.529]
P(H|lllll)  	= [ 0.000  0.001  0.078  0.296  0.624]


## MAP hypothesis

In [8]:
# h_map = argmax P(H | D) = argmax P(D | H) P(H)
# P(H | D) P(H) is above (P)
i = np.argmax(P)
print('MAP hypothesis: h%d' %(i+1))

MAP hypothesis: h5


## Prediction

Probability that next candy is lime

Using MAP hypothesis

In [10]:
PlhMAP = PdH['l'][i]
print('P(l|h_MAP) = %.3f' %(PlhMAP))

P(l|h_MAP) = 1.000


Using all hypotheses

In [11]:
cP = PdH['l'] * P
PlD = np.sum(cP)
print('P(l|D) = sum %s = %.3f' %(str(cP),PlD))

P(l|D) = sum [ 0.000  0.000  0.039  0.222  0.624] = 0.886


#Home Exercise

Di : outcome of rolling a 6-faces die

Z = D1 + D2 = sum of the outcomes of rolling 2 dice

S = D1 + D2 + D3 = sum of the outcomes of rolling 3 dice

Z in [2,12], S in [3,18]

**Question 1** 

Compute

Prior: P(S)  -- 16 values summing to 1

Posterior: P( S | D1 ) -- 16 x 6 matrix (each column sums to 1)

Posterior: P( S | D1, D2 ) -- 16 x 6 x 6 matrix (each column sums to 1)

Posterior: P( S | Z ) -- 16 x 12 matrix (each column sums to 1)

**Question 2** 

Verify experimentally that

P( S | Z, D1 ) = P ( S | Z )

